# DeBERTa / RoBERTa Fine-Tuning for Harmful vs Benign Conversations

This notebook fine-tunes an encoder-only transformer (DeBERTa or RoBERTa) as a **binary harm classifier** on your ContentID-style JSONL data.

- **Input format**: each row is a JSON object with `conversation` (list of `{role, message}`) and `label` (`0` = benign, `1` = harmful).
- **Objective**: classify a full conversation as **harmful** or **benign**.
- **Method**: standard supervised fine-tuning of `AutoModelForSequenceClassification` using Hugging Face `transformers`.

You can switch models by changing `BASE_MODEL_NAME` to e.g.:
- `"microsoft/deberta-v3-base"`
- `"microsoft/deberta-v3-large"`
- `"roberta-base"`
- `"roberta-large"`

In [ ]:
# Install dependencies (run once per environment)
# If you're running inside this repo's managed environment, you may not need this cell.
%pip install -U "transformers" "datasets" "evaluate" "scikit-learn" "mlflow"

In [ ]:
import os
import json
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import evaluate
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve


# -----------------------------
# Config: models & paths
# -----------------------------
# Change this to switch between DeBERTa / RoBERTa variants.
# Examples:
#   "microsoft/deberta-v3-base"
#   "microsoft/deberta-v3-large"
#   "roberta-base"
#   "roberta-large"
BASE_MODEL_NAME = "microsoft/deberta-v3-base"

# Optional: HF auth token for gated models
HF_TOKEN = os.getenv("HF_TOKEN", None)

# ContentID-style JSONL dataset paths
# Each line: {"conversation": [{"role": "user"|"assistant", "message": "..."}, ...], "label": 0|1, ...}
PROJECT_ROOT = Path(r"C:\Vijay\PyCode\ContentID").resolve()

DATASET_DIR = PROJECT_ROOT / "data" / "train"
TRAIN_JSONL = DATASET_DIR / "train.jsonl"
VAL_JSONL = DATASET_DIR / "val.jsonl"

# Output / logging / model paths
OUTPUT_ROOT = PROJECT_ROOT / "logs" / "outputs"
METRICS_ROOT = PROJECT_ROOT / "logs" / "metrics"
MLFLOW_ROOT = PROJECT_ROOT / "logs" / "experiments"
MODEL_ROOT = PROJECT_ROOT / "models"

RUN_NAME = BASE_MODEL_NAME.replace("/", "-") + "-contentid-seqcls"
OUTPUT_DIR = OUTPUT_ROOT / RUN_NAME
MODEL_DIR = MODEL_ROOT / RUN_NAME

for p in [OUTPUT_ROOT, METRICS_ROOT, MLFLOW_ROOT, MODEL_ROOT, OUTPUT_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

MAX_SEQ_LENGTH = 512  # adjust if you have more GPU memory
SEED = 42

# Reproducibility
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Using base model: {BASE_MODEL_NAME}")
print(f"Dataset dir: {DATASET_DIR}")
print(f"Train file: {TRAIN_JSONL}")
print(f"Val file:   {VAL_JSONL if VAL_JSONL.exists() else 'N/A'}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Model dir:  {MODEL_DIR}")
print(f"Metrics dir: {METRICS_ROOT}")
print(f"MLflow dir: {MLFLOW_ROOT}")

In [ ]:
def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file into a list of dicts."""
    data: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            data.append(json.loads(line))
    return data


def conversation_to_text(conversation: List[Dict[str, str]]) -> str:
    """Convert a list of {role, message} turns into a plain-text conversation."""
    lines: List[str] = []
    for turn in conversation:
        role = turn.get("role", "user").capitalize()
        msg = turn.get("message", "").strip()
        lines.append(f"{role}: {msg}")
    return "\n".join(lines)


def build_training_example(example: Dict[str, Any]) -> Dict[str, Any]:
    """Map a ContentID-style example to plain text + binary label.

    For encoder-only models we simply feed the raw conversation text; the
    harm policy is learned implicitly from the supervised labels.
    """
    conv = example.get("conversation") or []
    label = int(example.get("label", 0))

    text = conversation_to_text(conv)
    return {"text": text, "label": label}


def build_datasets(train_path: Path, val_path: Path | None = None) -> DatasetDict:
    assert train_path.exists(), f"Train JSONL not found: {train_path}"

    train_raw = load_jsonl(train_path)
    train_records = [build_training_example(row) for row in train_raw]

    dataset_dict: Dict[str, Dataset] = {}
    dataset_dict["train"] = Dataset.from_dict(
        {
            "text": [r["text"] for r in train_records],
            "label": [int(r["label"]) for r in train_records],
        }
    )

    if val_path is not None and val_path.exists():
        val_raw = load_jsonl(val_path)
        val_records = [build_training_example(row) for row in val_raw]
        dataset_dict["validation"] = Dataset.from_dict(
            {
                "text": [r["text"] for r in val_records],
                "label": [int(r["label"]) for r in val_records],
            }
        )

    ds = DatasetDict(dataset_dict)
    print(ds)
    return ds


datasets = build_datasets(TRAIN_JSONL, VAL_JSONL if VAL_JSONL.exists() else None)

In [ ]:
# -----------------------------
# Tokenizer, tokenization, and base model (sequence classification)
# -----------------------------

load_kwargs: Dict[str, Any] = {}
if HF_TOKEN is not None and HF_TOKEN != "":
    load_kwargs["token"] = HF_TOKEN

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME,
    use_fast=True,
    **load_kwargs,
)

# Ensure pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token is not None else tokenizer.cls_token

# For encoder models, right padding is standard
tokenizer.padding_side = "right"


def tokenize_function(batch: Dict[str, List[Any]]) -> Dict[str, Any]:
    encodings = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
    )
    encodings["labels"] = batch["label"]
    return encodings


print("Tokenizing datasets...")

tokenized_datasets = datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=["text", "label"],
)

print(tokenized_datasets)

print("Loading sequence classification base model (this can take a while)...")

config = AutoConfig.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=2,
    problem_type="single_label_classification",
    id2label={0: "BENIGN", 1: "HARMFUL"},
    label2id={"BENIGN": 0, "HARMFUL": 1},
    **load_kwargs,
)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    config=config,
    **load_kwargs,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded on:", device)

In [ ]:
# -----------------------------
# Metrics: accuracy, F1, ROC-AUC, AUPR, FPR@TPR 90/95
# -----------------------------

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, (tuple, list)):
        logits = logits[0]

    # Convert to numpy
    logits = np.array(logits)
    labels = np.array(labels).astype("int32")

    # Probabilities for harmful class (index 1)
    logits_shifted = logits - logits.max(axis=-1, keepdims=True)
    exp_logits = np.exp(logits_shifted)
    probs = exp_logits / exp_logits.sum(axis=-1, keepdims=True)
    harmful_probs = probs[:, 1]

    # Binary predictions at 0.5 threshold on harmful probability
    preds = (harmful_probs >= 0.5).astype("int32")

    metrics = {}
    metrics.update(accuracy_metric.compute(predictions=preds, references=labels))
    metrics.update(f1_metric.compute(predictions=preds, references=labels))

    # ROC-AUC and AUPR
    try:
        roc_auc = roc_auc_score(labels, harmful_probs)
    except ValueError:
        roc_auc = float("nan")

    try:
        aupr = average_precision_score(labels, harmful_probs)
    except ValueError:
        aupr = float("nan")

    # FPR at TPR 0.90 and 0.95
    try:
        fpr, tpr, _ = roc_curve(labels, harmful_probs)
        fpr_at_90 = float(fpr[tpr >= 0.90][0]) if np.any(tpr >= 0.90) else float("nan")
        fpr_at_95 = float(fpr[tpr >= 0.95][0]) if np.any(tpr >= 0.95) else float("nan")
    except ValueError:
        fpr_at_90 = float("nan")
        fpr_at_95 = float("nan")

    metrics.update(
        {
            "roc_auc": roc_auc,
            "aupr": aupr,
            "fpr_at_tpr_90": fpr_at_90,
            "fpr_at_tpr_95": fpr_at_95,
        }
    )

    return metrics

In [ ]:
# -----------------------------
# TrainingArguments, Trainer, and MLflow setup
# -----------------------------

import mlflow

# Use local file-based tracking under logs/experiments
tracking_uri = MLFLOW_ROOT.as_uri() if "MLFLOW_ROOT" in globals() else None
if tracking_uri is not None:
    mlflow.set_tracking_uri(tracking_uri)

MLFLOW_EXPERIMENT_NAME = "contentid_deberta_roberta_seqcls"
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print(f"MLflow tracking URI: {tracking_uri}")
print(f"MLflow experiment set to: {MLFLOW_EXPERIMENT_NAME}")


# Rough heuristic for steps per epoch (for logging/saving frequency)
train_batch_size = 8
gradient_accumulation_steps = 2

steps_per_epoch = max(
    1, len(tokenized_datasets["train"]) // (train_batch_size * gradient_accumulation_steps)
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    lr_scheduler_type="cosine",
    logging_steps=max(1, steps_per_epoch // 5),
    save_steps=steps_per_epoch,
    save_total_limit=3,
    evaluation_strategy="steps" if "validation" in tokenized_datasets else "no",
    eval_steps=steps_per_epoch if "validation" in tokenized_datasets else None,
    load_best_model_at_end="validation" in tokenized_datasets,
    metric_for_best_model="f1",
    greater_is_better=True,
    bf16=torch.cuda.is_available(),
    report_to=["mlflow"],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets.get("validation"),
    tokenizer=tokenizer,
    compute_metrics=compute_metrics if "validation" in tokenized_datasets else None,
)

print("Trainer ready.")

In [ ]:
# -----------------------------
# Train & save classifier
# -----------------------------

train_result = trainer.train()
print(train_result)

# Optional: final evaluation on validation set (if present)
eval_metrics = {}
if "validation" in tokenized_datasets:
    eval_metrics = trainer.evaluate()
    print("Final eval metrics:", eval_metrics)

# Save model + tokenizer to models/...
trainer.model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

# Save train/eval metrics to logs/metrics
metrics_payload = {
    "train": train_result.metrics,
    "eval": eval_metrics,
}
metrics_file = METRICS_ROOT / f"{RUN_NAME}_metrics.json"
with metrics_file.open("w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

print(f"Saved classifier to: {MODEL_DIR}")
print(f"Saved metrics to: {metrics_file}")
print("To load later, use AutoModelForSequenceClassification.from_pretrained(..., device_map='auto') on this folder.")

## Inference helper

The next cell defines a helper function that runs the trained `AutoModelForSequenceClassification`
on a list of conversations and returns logits, probabilities, and predicted labels.

In [ ]:
def classify_conversations(conversations: List[List[Dict[str, str]]]):
    """Run the trained classifier and return rich outputs for each conversation.

    For each input conversation, returns a dict with:
      - logits: raw scores for labels [0 (benign), 1 (harmful)]
      - softmax_probs: softmax probabilities over labels 0/1
      - harmful_prob: probability of the harmful class (index 1)
      - pred_label: final predicted label (0 or 1)
    """

    texts: List[str] = []
    for conv in conversations:
        texts.append(conversation_to_text(conv))

    enc = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=True,
    ).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(**enc)

    logits = outputs.logits  # [batch, 2]
    probs = torch.softmax(logits, dim=-1)

    harmful_probs = probs[:, 1]
    preds = torch.argmax(logits, dim=-1)

    results = []
    for i in range(logits.size(0)):
        results.append(
            {
                "logits": logits[i].cpu().tolist(),
                "softmax_probs": probs[i].cpu().tolist(),
                "harmful_prob": float(harmful_probs[i].cpu().item()),
                "pred_label": int(preds[i].cpu().item()),
            }
        )

    return results


# Example usage after training (uncomment and adapt):
# sample = load_jsonl(VAL_JSONL)[:2]
# convs = [ex["conversation"] for ex in sample]
# preds = classify_conversations(convs)
# preds[0]